# **Homework 1: COVID-19 Cases Prediction (Regression)**

Objectives:
* Solve a regression problem with deep neural networks (DNN).
* Understand basic DNN training tips.
* Familiarize yourself with PyTorch.

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

In [86]:
!nvidia-smi

Tue Oct 20 17:55:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.65                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              8W /  115W |     163MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Download data
If the Google Drive links below do not work, you can use the dropbox link below or download data from [Kaggle](https://www.kaggle.com/t/a339b77fa5214978bfb8dde62d3151fe), and upload data manually to the workspace.

In [87]:
# google drive link
# !pip install gdown
# !gdown --id '1BjXalPZxq9mybPKNjF3h5L3NcF7XKTS-' --output covid_train.csv
# !gdown --id '1B55t74Jg2E5FCsKCsUEkPKIuqaY7UIi1' --output covid_test.csv

# dropbox link
# !wget -O covid_train.csv https://www.dropbox.com/s/lmy1riadzoy0ahw/covid.train.csv?dl=0
# !wget -O covid_test.csv https://www.dropbox.com/s/zalbw42lu4nmhr2/covid.test.csv?dl=0

# Import packages

In [88]:
# Numerical Operations
import math
import numpy as np

# Reading/Writing Data
import pandas as pd
import os
import csv

# For Progress Bar
from tqdm import tqdm

# Pytorch
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# For plotting learning curve
from torch.utils.tensorboard import SummaryWriter

# Some Utility Functions

You do not need to modify this part.

In [89]:
def same_seed(seed): 
    '''Fixes random number generator seeds for reproducibility.'''
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_valid_split(data_set, valid_ratio, seed):
    '''Split provided training data into training set and validation set'''
    valid_set_size = int(valid_ratio * len(data_set)) 
    train_set_size = len(data_set) - valid_set_size
    train_set, valid_set = random_split(data_set, [train_set_size, valid_set_size], generator=torch.Generator().manual_seed(seed))
    return np.array(train_set), np.array(valid_set)

def predict(test_loader, model, device):
    model.eval() # Set your model to evaluation mode.
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)                        
        with torch.no_grad():                   
            pred = model(x)                     
            preds.append(pred.detach().cpu())   
    preds = torch.cat(preds, dim=0).numpy()  
    return preds

# Dataset

In [90]:
class COVID19Dataset(Dataset):
    '''
    x: Features.
    y: Targets, if none, do prediction.
    '''
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        else:
            return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)

# Neural Network Model
Try out different model architectures by modifying the class below.

In [91]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        # TODO: modify model's structure, be aware of dimensions. 
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            # nn.Dropout(0.5),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1) # (B, 1) -> (B)
        return x

# Feature Selection
Choose features you deem useful by modifying the function below.

In [92]:
def select_feat(train_data, valid_data, test_data, select_all=True):
    '''Selects useful features to perform regression'''
    y_train, y_valid = train_data[:,-1], valid_data[:,-1]
    raw_x_train, raw_x_valid, raw_x_test = train_data[:,:-1], valid_data[:,:-1], test_data

    if select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        feat_idx = list(range(1,117)) # TODO: Select suitable feature columns.
        # feat_idx = list(range(1, 38)) + [53, 69, 85, 101]
        # feat_idx = [53, 69, 85, 101] + [i for i in range(38, 117) if i not in (53, 69, 85, 101)]
        
    return raw_x_train[:,feat_idx], raw_x_valid[:,feat_idx], raw_x_test[:,feat_idx], y_train, y_valid

# Training Loop

In [93]:
def trainer(train_loader, valid_loader, model, config, device):

    criterion = nn.MSELoss(reduction='mean') # Define your loss function, do not modify this.

    # Define your optimization algorithm. 
    # TODO: Please check https://pytorch.org/docs/stable/optim.html to get more available algorithms.
    # TODO: L2 regularization (optimizer(weight decay...) or implement by your self).
    optimizer = torch.optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.7,
                                weight_decay=1e-2 ) 
    writer = SummaryWriter() # Writer of tensoboard.

    if not os.path.isdir('./models'):
        os.mkdir('./models') # Create directory of saving models.

    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train() # Set your model to train mode.
        loss_record = []

        # tqdm is a package to visualize your training progress.
        train_pbar = tqdm(train_loader, position=0, leave=True, disable=True)

        for x, y in train_pbar:
            optimizer.zero_grad()               # Set gradient to zero.
            x, y = x.to(device), y.to(device)   # Move your data to device. 
            pred = model(x)             
            loss = criterion(pred, y)
            loss.backward()                     # Compute gradient(backpropagation).
            optimizer.step()                    # Update parameters.
            step += 1
            loss_record.append(loss.detach().item())
            
            # Display current epoch number and loss on tqdm progress bar.
            train_pbar.set_description(f'Epoch [{epoch+1}/{n_epochs}]')
            train_pbar.set_postfix({'loss': loss.detach().item()})

        mean_train_loss = sum(loss_record)/len(loss_record)
        writer.add_scalar('Loss/train', mean_train_loss, step)

        model.eval() # Set your model to evaluation mode.
        loss_record = []
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)

            loss_record.append(loss.item())
            
        mean_valid_loss = sum(loss_record)/len(loss_record)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}]: Train loss: {mean_train_loss:.4f}, Valid loss: {mean_valid_loss:.4f}')
        writer.add_scalar('Loss/valid', mean_valid_loss, step)

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            torch.save(model.state_dict(), config['save_path']) # Save your best model
            # print('Saving model with loss {:.3f}...'.format(best_loss))
            early_stop_count = 0
        else: 
            early_stop_count += 1

        if early_stop_count >= config['early_stop']:
            print('\nModel is not improving, so we halt the training session.')
            break
    return best_loss

# Configurations
`config` contains hyper-parameters for training and the path to save your model.

In [94]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = {
    'seed': 5201314,      # Your seed number, you can pick your lucky number. :)
    'select_all': False,   # Whether to use all features.
    'valid_ratio': 0.2,   # validation_size = train_size * valid_ratio
    'n_epochs': 250,     # Number of epochs.            
    'batch_size': 256, 
    'learning_rate': 1e-5,              
    'early_stop': 100,    # If model has not improved for this many consecutive epochs, stop training.     
    'save_path': './models/model.ckpt'  # Your model will be saved here.
}

# Dataloader
Read data from files and set up training, validation, and testing sets. You do not need to modify this part.

In [95]:
# Set seed for reproducibility
same_seed(config['seed'])


# train_data size: 3009 x 89 (35 states + 18 features x 3 days) 
# test_data size: 997 x 88 (without last day's positive rate)
train_data, test_data = pd.read_csv('./covid_train.csv').values, pd.read_csv('./covid_test.csv').values
train_data, valid_data = train_valid_split(train_data, config['valid_ratio'], config['seed'])

# Print out the data size.
print(f"""train_data size: {train_data.shape} 
valid_data size: {valid_data.shape} 
test_data size: {test_data.shape}""")

# Select features
x_train, x_valid, x_test, y_train, y_valid = select_feat(train_data, valid_data, test_data, config['select_all'])

# Print out the number of features.
print(f'number of features: {x_train.shape[1]}')

# ============ 特征标准化（用 train 的统计量）============
mean = x_train.mean(axis=0)
std  = x_train.std(axis=0) + 1e-8          # +1e-8 防止除 0

x_train = ((x_train - mean) / std).astype(np.float32)
x_valid = ((x_valid - mean) / std).astype(np.float32)
x_test  = ((x_test  - mean) / std).astype(np.float32)

print(f'标准化后：mean={x_train.mean():.4f}  std={x_train.std():.4f}')

train_dataset, valid_dataset, test_dataset = COVID19Dataset(x_train, y_train), \
                                            COVID19Dataset(x_valid, y_valid), \
                                            COVID19Dataset(x_test)

# Pytorch data loader loads pytorch dataset into batches.
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=config['batch_size'], shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, pin_memory=True)

train_data size: (2160, 118) 
valid_data size: (539, 118) 
test_data size: (1078, 117)
number of features: 116
标准化后：mean=-0.0000  std=1.0000


# Start training!

In [96]:
model = My_Model(input_dim=x_train.shape[1]).to(device) # put your model and data on the same computation device.
trainer(train_loader, valid_loader, model, config, device)

Epoch [1/250]: Train loss: 126.6731, Valid loss: 139.4007
Epoch [10/250]: Train loss: 126.2927, Valid loss: 133.6724
Epoch [20/250]: Train loss: 121.9565, Valid loss: 131.0049
Epoch [30/250]: Train loss: 116.2083, Valid loss: 128.5405
Epoch [40/250]: Train loss: 113.9244, Valid loss: 117.1973
Epoch [50/250]: Train loss: 107.6653, Valid loss: 112.0982
Epoch [60/250]: Train loss: 96.4684, Valid loss: 100.5246
Epoch [70/250]: Train loss: 82.1745, Valid loss: 78.7607
Epoch [80/250]: Train loss: 61.7739, Valid loss: 63.4882
Epoch [90/250]: Train loss: 45.9190, Valid loss: 48.7093
Epoch [100/250]: Train loss: 35.6844, Valid loss: 37.1178
Epoch [110/250]: Train loss: 31.4677, Valid loss: 32.2963
Epoch [120/250]: Train loss: 28.7307, Valid loss: 30.1427
Epoch [130/250]: Train loss: 26.7731, Valid loss: 24.0575
Epoch [140/250]: Train loss: 25.1045, Valid loss: 24.9705
Epoch [150/250]: Train loss: 23.3386, Valid loss: 24.9391
Epoch [160/250]: Train loss: 21.9025, Valid loss: 23.1139
Epoch [170/2

10.102103551228842

# Testing
The predictions of your model on testing set will be stored at `pred.csv`.

In [97]:
def save_pred(preds, file):
    ''' Save predictions to specified file '''
    with open(file, 'w') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'tested_positive'])
        for i, p in enumerate(preds):
            writer.writerow([i, p])

model = My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path']))
preds = predict(test_loader, model, device) 
save_pred(preds, 'submission.csv')         

100%|██████████| 5/5 [00:00<00:00, 588.84it/s]


# Download

Run this block to download the `pred.csv` by clicking.

In [98]:
from IPython.display import FileLink
FileLink(r'submission.csv')


/home/cc/projects/hw01/submission.csv

In [ ]:
# import pandas as pd
# sub = pd.read_csv("submission.csv")
# print(sub.shape)
# print(sub.head())
for lr in [1e-5, 1e-4, 1e-3, 1e-2]:
    same_seed(config['seed'])
    model = My_Model(input_dim=x_train.shape[1]).to(device)   # 每次重建模型
    config['learning_rate'] = lr
    best = trainer(train_loader, valid_loader, model, config, device)
    print(f'>>> lr={lr:.0e}   best valid loss = {best:.5f}\n')


Epoch [1/250]: Train loss: 118.7354, Valid loss: 105.9637
Epoch [10/250]: Train loss: 3.7535, Valid loss: 4.3334
Epoch [20/250]: Train loss: 2.2188, Valid loss: 2.5964
Epoch [30/250]: Train loss: 1.6982, Valid loss: 1.9523
Epoch [40/250]: Train loss: 1.4610, Valid loss: 1.8293
Epoch [50/250]: Train loss: 1.2904, Valid loss: 1.9837
Epoch [60/250]: Train loss: 1.2019, Valid loss: 1.5566
Epoch [70/250]: Train loss: 1.1117, Valid loss: 1.3076
Epoch [80/250]: Train loss: 1.0540, Valid loss: 1.3813
Epoch [90/250]: Train loss: 1.0012, Valid loss: 1.3573
Epoch [100/250]: Train loss: 0.9784, Valid loss: 1.2541
Epoch [110/250]: Train loss: 0.9334, Valid loss: 1.3097
Epoch [120/250]: Train loss: 0.9211, Valid loss: 1.3144
Epoch [130/250]: Train loss: 0.9221, Valid loss: 1.4967
Epoch [140/250]: Train loss: 0.8945, Valid loss: 1.3316
Epoch [150/250]: Train loss: 0.8855, Valid loss: 1.1598
Epoch [160/250]: Train loss: 0.8475, Valid loss: 1.1797
Epoch [170/250]: Train loss: 0.8500, Valid loss: 1.4934